# Landslide hazard step 01: combined-class intersections

Duplicates the source-and-runout intersection workflow for the 15 combined-class rasters (three scenarios and five return periods). Each raster is reprojected to Jamaica CRS (`EPSG:3448`) before running vector-raster intersections. All outputs use a separate `combined_class` namespace so the original results are preserved.


In [ ]:
import re
import subprocess
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')

landslide_rasters_root = base_path / 'dphil_paper_3/inputs/landslides/landslide_rp_susceptibility_maps/combined_class'
networks_data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
networks_metadata_csv = networks_data_root / 'networks/network_layers_hazard_intersections_details.csv'

output_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/landslide_network_intersections_combined_class'
projected_rasters_path = output_path / 'projected_rasters_epsg3448_combined_class'

vector_intersections_script = base_path / 'robyns_libraries/vector_raster_intersections.py'

jamaica_metric_grid_crs = 'EPSG:3448'

output_path.mkdir(parents=True, exist_ok=True)
projected_rasters_path.mkdir(parents=True, exist_ok=True)

print('Landslide combined-class rasters root:', landslide_rasters_root)
print('Networks metadata csv:', networks_metadata_csv)
print('Output path:', output_path)


In [ ]:
scenario_lookup = {
    'combined_class_baseline': 'baseline',
    'combined_class_deforestation': 'deforestation',
    'combined_class_reafforestation': 'reafforestation',
}

return_period_pattern = re.compile(r'_rp(\d+)\.tif$', flags=re.IGNORECASE)

def reproject_raster_to_epsg3448(source_raster: Path, destination_raster: Path, destination_crs: str = 'EPSG:3448'):
    with rasterio.open(source_raster) as source_dataset:
        transform, width, height = calculate_default_transform(
            source_dataset.crs, destination_crs, source_dataset.width, source_dataset.height, *source_dataset.bounds
        )

        destination_profile = source_dataset.profile.copy()
        destination_profile.update(
            crs=destination_crs,
            transform=transform,
            width=width,
            height=height,
        )

        with rasterio.open(destination_raster, 'w', **destination_profile) as destination_dataset:
            for band_index in range(1, source_dataset.count + 1):
                reproject(
                    source=rasterio.band(source_dataset, band_index),
                    destination=rasterio.band(destination_dataset, band_index),
                    src_transform=source_dataset.transform,
                    src_crs=source_dataset.crs,
                    dst_transform=transform,
                    dst_crs=destination_crs,
                    src_nodata=source_dataset.nodata,
                    dst_nodata=source_dataset.nodata,
                    resampling=Resampling.nearest,
                )

hazard_rows = []

for tif_path in sorted(landslide_rasters_root.rglob('*.tif')):
    scenario_folder_name = tif_path.parent.name
    if scenario_folder_name not in scenario_lookup:
        continue

    scenario_name = scenario_lookup[scenario_folder_name]
    return_period_match = return_period_pattern.search(tif_path.name)
    if not return_period_match:
        raise ValueError(f'Could not parse RP from filename: {tif_path.name}')

    return_period_years = int(return_period_match.group(1))

    projected_filename = f'landslide_{scenario_name}_rp_{return_period_years}_epsg3448_combined_class.tif'
    projected_raster = projected_rasters_path / projected_filename

    reproject_raster_to_epsg3448(tif_path, projected_raster, jamaica_metric_grid_crs)

    hazard_rows.append({
        'hazard': 'landslide_combined_class',
        'scenario': scenario_name,
        'rp': return_period_years,
        'key': f'landslide_combined_class_{scenario_name}_rp_{return_period_years}',
        'path': str(projected_raster),
        'fname': str(projected_raster),
        'source_path': str(tif_path),
    })

hazard_layers_table = pd.DataFrame(hazard_rows).sort_values(['scenario', 'rp']).reset_index(drop=True)

scenario_counts = hazard_layers_table.groupby('scenario').size().to_dict()
scenario_return_periods = {
    scenario_name: sorted(group['rp'].astype(int).tolist())
    for scenario_name, group in hazard_layers_table.groupby('scenario')
}
print('Hazard rasters found by scenario:', scenario_counts)
print('Return periods found by scenario:', scenario_return_periods)
print('Total hazard rasters:', len(hazard_layers_table))

expected_scenarios = {'baseline', 'deforestation', 'reafforestation'}
expected_return_periods = [5, 10, 25, 50, 100]
if set(scenario_counts.keys()) != expected_scenarios:
    raise ValueError(f'Expected scenarios {expected_scenarios} but found {set(scenario_counts.keys())}')

for scenario_name in expected_scenarios:
    found_return_periods = scenario_return_periods.get(scenario_name, [])
    if found_return_periods != expected_return_periods:
        raise ValueError(
            f'Expected return periods {expected_return_periods} for {scenario_name}, found {found_return_periods}. '
            'Update expected_return_periods if new combined-class rasters are intentionally added.'
        )

hazard_layers_output_file = output_path / 'landslide_rasters_for_intersections_combined_class.csv'
hazard_layers_table.to_csv(hazard_layers_output_file, index=False)

hazard_layers_table


In [ ]:
network_layers_table = pd.read_csv(networks_metadata_csv)
network_layers_table = network_layers_table[['path']].drop_duplicates().reset_index(drop=True)

# Same path fix used in the coastal workflow so paths resolve from common_incoming_data
network_layers_table['path'] = network_layers_table['path'].str.replace(
    r'^networks/',
    'networks/networks/',
    regex=True,
)

network_layers_output_file = output_path / 'network_layers_for_intersections_combined_class.csv'
network_layers_table.to_csv(network_layers_output_file, index=False)

print('Network layers file:', network_layers_output_file)
print('Hazard layers file:', hazard_layers_output_file)
print('Network files (unique gpkg count):', len(network_layers_table))

network_layers_table


In [ ]:
run_intersections = True  # Set to True to run vector-raster intersections
force_refresh_intersections = True  # Delete existing combined-class intersection parquet files before rerun.

if run_intersections:
    hazard_slug = hazard_layers_output_file.stem
    if force_refresh_intersections:
        stale_outputs = sorted(output_path.glob(f'*_splits__{hazard_slug}__*.geoparquet'))
        for stale_output in stale_outputs:
            stale_output.unlink()
        print(f'Removed {len(stale_outputs)} existing combined-class intersection outputs before rerun.')

    command_arguments = [
        'python',
        str(vector_intersections_script),
        str(network_layers_output_file),
        str(hazard_layers_output_file),
        str(output_path),
    ]
    print('* Start the processing of vector-raster intersections')
    print(command_arguments)
    subprocess.run(command_arguments, check=True)

print('* Done with the processing of vector-raster intersections')


## Completion Check

Run this after intersections to confirm all expected network outputs were produced.

In [ ]:
checks = {
    'roads': [
        'roads_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
        'roads_splits__landslide_rasters_for_intersections_combined_class__edges.geoparquet',
    ],
    'rail': [
        'rail_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
        'rail_splits__landslide_rasters_for_intersections_combined_class__edges.geoparquet',
    ],
    'ports': [
        'port_polygon_splits__landslide_rasters_for_intersections_combined_class__areas.geoparquet',
    ],
    'airports': [
        'airport_polygon_splits__landslide_rasters_for_intersections_combined_class__areas.geoparquet',
    ],
    'water_irrigation': [
        'irrigation_assets_NIC_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
        'irrigation_assets_NIC_splits__landslide_rasters_for_intersections_combined_class__edges.geoparquet',
    ],
    'water_pipelines': [
        'pipelines_NWC_splits__landslide_rasters_for_intersections_combined_class__edges.geoparquet',
    ],
    'water_potable': [
        'potable_facilities_NWC_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
    ],
    'water_wastewater': [
        'waste_water_facilities_NWC_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
    ],
    'energy': [
        'electricity_network_v3.1_splits__landslide_rasters_for_intersections_combined_class__nodes.geoparquet',
        'electricity_network_v3.1_splits__landslide_rasters_for_intersections_combined_class__edges.geoparquet',
    ],
    'buildings': [
        'buildings_assigned_economic_activity_splits__landslide_rasters_for_intersections_combined_class__areas.geoparquet',
    ],
}

summary_rows = []
for group_name, filenames in checks.items():
    group_ok = True
    for filename in filenames:
        file_path = output_path / filename
        exists = file_path.exists()
        row_count = None
        hazard_column_count = None

        if exists:
            intersection_data = gpd.read_parquet(file_path)
            row_count = len(intersection_data)
            hazard_columns = [
                column_name
                for column_name in intersection_data.columns
                if column_name.startswith('landslide_combined_class_') and '_rp_' in column_name
            ]
            hazard_column_count = len(hazard_columns)
        else:
            group_ok = False

        summary_rows.append({
            'group': group_name,
            'file': filename,
            'exists': exists,
            'rows': row_count,
            'hazard_columns': hazard_column_count,
        })

    print(f"{group_name}: {'OK' if group_ok else 'MISSING'}")

summary_df = pd.DataFrame(summary_rows)
summary_df
